<a href="https://colab.research.google.com/github/Hidrovales/ManejoClima/blob/main/Balanco_Hidrico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> Atualize o link acima com o caminho real deste notebook dentro do repositório, quando ele for enviado ao GitHub.

# Balanço Hídrico com dados da Open-Meteo (Google Colab)

Este notebook usa `main.py` (busca de dados na Open-Meteo), `Calcula_ETo.py` (cálculo do ETo) e `Balanco_Hidrico.py` (simulação do balanço hídrico), do repositório [Hidrovales/ManejoClima](https://github.com/Hidrovales/ManejoClima), para:

1. Buscar as coordenadas do local;
2. Registrar o dia de hoje na série acumulada (rodar uma vez por dia);
3. Acompanhar a série que vai crescendo com o tempo;
4. Rodar o balanço hídrico com o que já foi acumulado;
5. **Visualizar os gráficos** gerados por `Balanco_Hidrico.py`.

> Rode a célula de "Registrar o dia de hoje" uma vez por dia — é isso que alimenta a série aos poucos.

> **Feito para rodar no Google Colab.** O disco do Colab é temporário (some quando a sessão reinicia), por isso este notebook clona o código do GitHub a cada execução e guarda o banco de dados (que precisa persistir entre os dias) no Google Drive.

## 1. Clonar o repositório com o código

In [ ]:
!git clone https://github.com/Hidrovales/ManejoClima.git
%cd ManejoClima

## 2. Importar os módulos e montar o Google Drive

O Drive é necessário para o banco (`.db`) persistir entre sessões — sem isso, a série acumulada dia a dia se perderia toda vez que o Colab reiniciasse.

In [ ]:
%matplotlib inline

import main
import Calcula_ETo as gse
import Balanco_Hidrico as bh

from google.colab import drive
import os

drive.mount("/content/drive")
PASTA_DRIVE = "/content/drive/MyDrive/Hidrovales/ManejoClima/dados"
os.makedirs(PASTA_DRIVE, exist_ok=True)
DATABASE_PATH = f"{PASTA_DRIVE}/balanco_hidrico.db"
print("Banco de dados em:", DATABASE_PATH)

## 3. Buscar coordenadas do local

In [ ]:
NOME_CIDADE = "Rio Pardo de Minas, Brasil"

LOCAL = local = main.buscar_coordenadas(NOME_CIDADE)
local

## 4. Registrar o dia de hoje

Busca os dados diários de hoje na Open-Meteo, calcula o ETo e grava (ou atualiza, se já rodou hoje) uma linha na série acumulada. **Rode essa célula uma vez por dia** para a série crescer.

In [ ]:
resultado_hoje = bh.registrar_dia(
    local=local["nome"],
    latitude=local["latitude"],
    longitude=local["longitude"],
    altitude=local["altitude"],
    database_path=DATABASE_PATH,
    timezone=local.get("timezone", "auto"),
)
resultado_hoje

## 5. Ver a série acumulada até agora

In [ ]:
serie = bh.carregar_serie_diaria(local["nome"], DATABASE_PATH)
serie

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(serie["DATA"], serie["ETO"], "o-", color="tab:red", label="ETo (mm)")
ax1.set_ylabel("ETo (mm)", color="tab:red")
ax1.tick_params(axis="y", labelcolor="tab:red")

ax2 = ax1.twinx()
ax2.bar(serie["DATA"], serie["PRECIPITACAO"], color="tab:blue", alpha=0.4, label="Precipitação (mm)")
ax2.set_ylabel("Precipitação (mm)", color="tab:blue")
ax2.tick_params(axis="y", labelcolor="tab:blue")

plt.title("Série acumulada dia a dia")
fig.autofmt_xdate(rotation=45)
plt.show()

## 6. Rodar o balanço hídrico

Ajuste os parâmetros da cultura abaixo. Se a série ainda não cobrir o ciclo inteiro, `balanco()` avisa e calcula até onde já há dado — sem quebrar.

In [ ]:
CULTURA = "Milho"

periodo = {"inicial": 20, "desenvolvimento": 30, "media": 40, "final": 20}  # dias de cada fase
kc_etapas = {"inicial": 0.3, "media": 1.1, "final": 0.6}
forma_kc = {"inicial": True, "desenvolvimento": False, "media": True, "final": False}
z_etapas = {"inicial": 0.2, "media": 0.6, "final": 0.6}   # profundidade radicular [m]
forma_z = {"inicial": True, "desenvolvimento": False, "media": True, "final": True}

theta_fc = 0.23   # capacidade de campo [m3/m3]
theta_wp = 0.10   # ponto de murcha [m3/m3]
p = 0.5           # fator de disponibilidade hídrica

data_in = {"ano": 2026, "mes": 9, "dia": 25}  # data de início do cultivo

eto_df = serie[["DATA", "ETO"]]
p_df = serie[["DATA", "PRECIPITACAO"]]

bh.balanco(
    local=local["nome"], cultura=CULTURA,
    theta_fc=theta_fc, theta_wp=theta_wp, p=p,
    P=p_df, eto=eto_df,
    periodo=periodo, z_etapas=z_etapas, forma_z=forma_z,
    kc_etapas=kc_etapas, forma_kc=forma_kc,
    data_in=data_in, database_path=DATABASE_PATH,
)

## 7. Visualizar os gráficos

In [ ]:
resultado = bh.carregar_resultado(local["nome"], CULTURA, DATABASE_PATH)
bh.plot_extras(resultado, figsize=(9, 4))

In [ ]:
bh.plot_balanco(resultado, figsize=(9, 4))